# 04 — SQL Avançado e Otimização

**Objetivos:** usar CTE, funções de janela e plano de execução de forma interpretável

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia técnico — SQL avançado e otimização](./reference_materials/04_sql_avancado_otimizacao.pdf)
- [Cheat sheet — SQL avançado e otimização](./cheat_sheets/cheat_sheet_sql_avancado_otimizacao.pdf)
- [Documentação e arquivos de resultados](./README.pt-BR.md)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## CTE e consulta analítica

CTEs ajudam a nomear etapas lógicas; não garantem desempenho superior por si mesmas.

In [2]:
import sqlite3
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys=ON')
conn.executescript((Path('sql')/'01_schema_sqlite.sql').read_text(encoding='utf8'))
leituras = pd.read_csv(DATA/'leituras_sensores.csv')
leituras.to_sql('leituras',conn,if_exists='append',index=False)
print('SQLite inicializado:',conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0], 'leituras')


SQLite inicializado: 72 leituras


## Funções de janela

`ROW_NUMBER` e médias móveis preservam cada linha e acrescentam contexto por partição. `GROUP BY` reduziria a quantidade de linhas.

In [3]:
q='''WITH ranked AS (SELECT id_sensor,data_hora,temperatura_c, ROW_NUMBER() OVER(PARTITION BY id_sensor ORDER BY temperatura_c DESC) pos FROM leituras) SELECT * FROM ranked WHERE pos<=2 ORDER BY id_sensor,pos''';display(pd.read_sql_query(q,conn))

,id_sensor,data_hora,temperatura_c,pos
0,1,2025-01-02 22:00:00,21.74,1
1,1,2025-01-01 22:00:00,21.54,2
2,2,2025-01-03 08:00:00,23.69,1
3,2,2025-01-02 14:00:00,23.67,2
4,3,2025-01-03 03:00:00,25.39,1
5,3,2025-01-03 09:00:00,25.31,2


## Média móvel de três observações

O quadro inclui a observação corrente e duas anteriores por sensor; nas duas primeiras, a janela é menor.

In [4]:
display(pd.read_sql_query('''SELECT id_sensor,data_hora,temperatura_c, ROUND(AVG(temperatura_c) OVER (PARTITION BY id_sensor ORDER BY data_hora ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) AS mm3 FROM leituras ORDER BY id_sensor,data_hora LIMIT 9''',conn))

,id_sensor,data_hora,temperatura_c,mm3
0,1,2025-01-01 10:00:00,21.21,21.21
1,1,2025-01-01 13:00:00,19.66,20.44
2,1,2025-01-01 16:00:00,19.66,20.18
3,1,2025-01-01 19:00:00,20.02,19.78
4,1,2025-01-01 22:00:00,21.54,20.41
5,1,2025-01-02 01:00:00,20.14,20.57
6,1,2025-01-02 04:00:00,21.11,20.93
7,1,2025-01-02 07:00:00,19.30,20.18
8,1,2025-01-02 10:00:00,19.80,20.07


## Índices e plano de execução

EXPLAIN QUERY PLAN mostra a estratégia escolhida pelo SQLite; um índice não é sempre benéfico e deve responder a consultas reais.

In [5]:
query='SELECT * FROM leituras WHERE id_sensor=2 ORDER BY data_hora'; print('Antes:');display(pd.read_sql_query('EXPLAIN QUERY PLAN '+query,conn));conn.execute('CREATE INDEX IF NOT EXISTS idx_leituras_sensor_data ON leituras(id_sensor,data_hora)');print('Depois:');display(pd.read_sql_query('EXPLAIN QUERY PLAN '+query,conn))

Antes:


,id,parent,notused,detail
0,3,0,0,SCAN leituras
1,17,0,0,USE TEMP B-TREE FOR ORDER BY


Depois:


,id,parent,notused,detail
0,4,0,0,SEARCH leituras USING INDEX idx_leituras_senso...


## Exercício resolvido — último valor por sensor

Usar ROW_NUMBER com data decrescente e isolar pos=1 para cada sensor. Se houver empates de timestamp, adicionar critério determinístico.

In [6]:
display(pd.read_sql_query('''WITH ult AS (SELECT id_sensor,data_hora,temperatura_c,ROW_NUMBER() OVER(PARTITION BY id_sensor ORDER BY data_hora DESC,id_evento DESC) rn FROM leituras) SELECT id_sensor,data_hora,temperatura_c FROM ult WHERE rn=1 ORDER BY id_sensor''',conn))

,id_sensor,data_hora,temperatura_c
0,1,2025-01-04 07:00:00,19.80
1,2,2025-01-04 05:00:00,23.00
2,3,2025-01-04 06:00:00,23.83


## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Índices e plano de consulta

Use `EXPLAIN QUERY PLAN` para verificar a escolha do otimizador. Um índice pode acelerar filtros seletivos, mas a vantagem depende de cardinalidade, distribuição, atualização e tamanho da tabela. **Não confunda plano de execução com prova de ganho de desempenho**; benchmarks precisam de cargas comparáveis e várias medições.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [7]:
import sqlite3
with sqlite3.connect(':memory:') as db:
    db.execute('CREATE TABLE eventos(id INTEGER PRIMARY KEY, sensor INTEGER, valor REAL)')
    db.executemany('INSERT INTO eventos VALUES (?,?,?)',[(i,i%20,float(i%7)) for i in range(1000)])
    antes=db.execute('EXPLAIN QUERY PLAN SELECT * FROM eventos WHERE sensor=7').fetchall()
    db.execute('CREATE INDEX idx_eventos_sensor ON eventos(sensor)')
    depois=db.execute('EXPLAIN QUERY PLAN SELECT * FROM eventos WHERE sensor=7').fetchall()
    print('Sem índice:',antes)
    print('Com índice:',depois)
    assert sum(1 for x in db.execute('SELECT * FROM eventos WHERE sensor=7'))==50

Sem índice: [(2, 0, 0, 'SCAN eventos')]
Com índice: [(3, 0, 0, 'SEARCH eventos USING INDEX idx_eventos_sensor (sensor=?)')]


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [8]:
pd.read_sql_query('''WITH ult AS (SELECT id_sensor,data_hora,temperatura_c,ROW_NUMBER() OVER(PARTITION BY id_sensor ORDER BY data_hora DESC,id_evento DESC) rn FROM leituras) SELECT id_sensor,data_hora,temperatura_c FROM ult WHERE rn=1 ORDER BY id_sensor''',conn).to_csv(OUT / '04_ultima_leitura_por_sensor.csv',index=False)
print('Exportado: outputs/04_ultima_leitura_por_sensor.csv')

Exportado: outputs/04_ultima_leitura_por_sensor.csv
